### Auto-Scrabble – Lösung Schritt für Schritt

*36. Bundeswettbewerb Informatik (2017), 1. Runde, Aufgabe 4*

Familie Petersen fährt mit dem Auto aus dem Urlaub nach Hause. Plötzlich ruft die kleine Heike: „Da steht mein Naaaame, daaaa auf dem Autoschild!“ Ganz begeistert zeigt sie auf das Kennzeichen, auf dem zu lesen ist: `HEI-KE`.

Ihr Bruder Timo ist nicht beeindruckt. „Es gibt doch so viele verschiedene Kennzeichen. Da kann man bestimmt jedes Wort schreiben.“ Heike antwortet: „Neee, dein Name geht doch gar nicht!“

Wer hat Recht? Ein deutsches KfZ-Kennzeichen besteht aus drei Teilen:

- einem Kürzel für eine Stadt oder einen Kreis aus der Liste, die du auf den BwInf-Webseiten findest;
- einem Mittelteil, der aus einem oder zwei Buchstaben besteht (ohne Umlaute) und
- einer Zahl, die wir hier ignorieren.

#### Aufgabe 4

1. Stimmt es, dass TIMO nicht auf einem Kennzeichen stehen kann?
2. Gib ein weiteres Wort mit vier Buchstaben an, das nicht auf einem Kennzeichen stehen kann. Findest du sogar ein solches Wort mit drei Buchstaben? Mit zwei?
3. Schreibe ein Programm, das ein Wort einliest und überprüft, ob es mit mehreren Kennzeichen geschrieben werden kann. Dabei darf das Wort aufgeteilt werden, z. B. `B-UN`, `DE-SW`, `E-TT`, `B-E`, `WER-B` für BUNDESWETTBEWERB (die Zahlen lassen wir weg). Ein Kennzeichen muss immer voll ausgenutzt werden. Folgendes ist also nicht erlaubt: `IN-FO`, `RM-AT`, `IK-AA` für INFORMATIK (hinter dem `K` bliebe vom Kennzeichen `IK-AA` das `A` übrig).
4. Erweitere dein Programm so, dass es außerdem eine Folge von Kennzeichen ausgibt, mit denen das Wort gebildet werden kann (wenn es eine gibt).

Wende dein Programm auf die Beispiele an, die du auf den BwInf-Webseiten findest.

#### Eingabeformat

- `beispieldaten/kuerzelliste.txt`: die Liste der Kürzel für Städte und Kreise, ein Kürzel pro Zeile (684 Stück, ein bis drei Buchstaben).
- `beispieldaten/autoscrabble.txt`: die zu prüfenden Wörter, ein Wort pro Zeile (BIBER, BUNDESWETTBEWERB, …). Manche Wörter enthalten Umlaute.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zerlegen in kleine Teilprobleme:** Statt das ganze Wort auf einmal aufzuteilen, fragen wir nur: „Bis zu welcher Stelle kommt man mit Kennzeichen, wenn man von vorne anfängt?“ Jede Stelle, die man erreicht, ist Ausgangspunkt für das nächste Kennzeichen.
- **Dynamische Programmierung (Tabelle):** Wir legen eine Tabelle an, die zu jeder Stelle `i` des Wortes sagt, ob die ersten `i` Buchstaben aus Kennzeichen bestehen können, und wenn ja, wie. Jede Stelle wird nur einmal bearbeitet. Ohne Tabelle würde man dieselben Teilwörter immer wieder neu aufteilen, und das wächst für lange Wörter stark an (bei jeder Stelle mehrere Möglichkeiten, die sich multiplizieren).
- **Nur die Länge zählt:** Ob ein Kennzeichen `AB-C` oder `A-BC` lautet, ist für den Rest des Wortes egal, denn beide verbrauchen drei Buchstaben. Daher reicht pro Stelle *eine* Aufteilung.
- **Nachschlagen in einer Menge:** Ob ein Textstück ein Kürzel ist, fragt man mit `in` an einer `set`-Menge. Das geht viel schneller als in einer Liste.

**Nützliche Funktionen und Bibliotheken**
- Ausschnitte von Texten: `wort[i:i+3]` (die Buchstaben ab Stelle `i`).
- `set(...)` für schnelles Nachschlagen, `split()` zum Zerlegen der Datei in Zeilen.
- `itertools.product` (nur im Abschnitt zu den unmöglichen Wörtern), um alle Wörter einer Länge durchzugehen.

## Die Idee

Ein Wort besteht aus Kennzeichen, wenn man es von vorne in Stücke schneiden kann, die jeweils so aussehen: **Kürzel (1 bis 3 Buchstaben) + Mittelteil (1 oder 2 Buchstaben, keine Umlaute)**.

Wir bauen eine Tabelle `tabelle[0..n]` für ein Wort mit `n` Buchstaben:

- `tabelle[i]` enthält eine Liste von Kennzeichen, mit denen die ersten `i` Buchstaben geschrieben werden können, oder `None`, wenn das nicht geht.
- `tabelle[0]` ist die leere Liste (null Buchstaben brauchen null Kennzeichen).
- Wir gehen `i = 0, 1, 2, …` durch. Ist `tabelle[i]` nicht `None`, suchen wir alle Kennzeichen, die ab Buchstabe `i` passen. Für jedes, das bis zur Stelle `e` reicht, tragen wir in `tabelle[e]` die Liste von `tabelle[i]` plus dieses Kennzeichen ein (falls dort noch nichts steht).
- Am Ende steht in `tabelle[n]` eine Aufteilung des ganzen Wortes oder `None`.

Weil jede Stelle nur einmal bearbeitet wird und von dort höchstens `3 · 2 = 6` Kennzeichen ausgehen, ist das Verfahren auch für sehr lange Wörter schnell. Das ist genau die Idee der offiziellen Lösungshinweise (Abbildung dort für das Wort BWINF).

## Schritt 1: Daten einlesen

Die Kürzel kommen in eine **Menge** (`set`), damit das Nachschlagen schnell geht. Die Dateien sind UTF-8 (wegen der Umlaute in den Wörtern).

In [ ]:
def lies_woerter(dateiname):
    """Gibt die Zeilen der Datei als Liste von Wörtern zurück."""
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read().split()


kuerzel = set(lies_woerter('beispieldaten/kuerzelliste.txt'))
woerter = lies_woerter('beispieldaten/autoscrabble.txt')

laengste = max(len(k) for k in kuerzel)
print(len(kuerzel), 'Kürzel, die längsten haben', laengste, 'Buchstaben')
print('Beispiele:', sorted(kuerzel)[:12])
print(len(woerter), 'Wörter:', woerter)

684 Kürzel, die längsten haben 3 Buchstaben
Beispiele: ['A', 'AA', 'AB', 'ABG', 'ABI', 'AC', 'AE', 'AH', 'AIB', 'AIC', 'AK', 'ALF']
12 Wörter: ['BIBER', 'BUNDESWETTBEWERB', 'CLINTON', 'DONAUDAMPFSCHIFFFAHRTSKAPITÄNSMÜTZE', 'ETHERNET', 'INFORMATIK', 'LLANFAIRPWLLGWYNGYLLGOGERYCHWYRNDROBWLLLLANTYSILIOGOGOGOCH', 'RINDFLEISCHETIKETTIERUNGSÜBERWACHUNGSAUFGABENÜBERTRAGUNGSGESETZ', 'SOFTWARE', 'TRUMP', 'TSCHÜSS', 'VERKEHRSWEGEPLANUNGSBESCHLEUNIGUNGSGESETZ']


## Schritt 2: Welche Kennzeichen passen ab Stelle i?

Ein Kennzeichen ab Stelle `i` besteht aus einem Kürzel, das dort beginnt, und dem Mittelteil mit einem oder zwei Buchstaben. Der Mittelteil darf **keine Umlaute** enthalten (nur `A` bis `Z`), das Wort darf nicht zu Ende sein, bevor das Kennzeichen vollständig ist.

Die Funktion gibt für jedes passende Kennzeichen ein Tupel `(ende, kuerzel, mitte)` zurück. `ende` ist die Stelle, an der das nächste Kennzeichen beginnen müsste.

In [ ]:
BUCHSTABEN = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'


def ist_mitte(text):
    """True, wenn text nur aus Buchstaben A bis Z besteht (keine Umlaute)."""
    for zeichen in text:
        if zeichen not in BUCHSTABEN:
            return False
    return True


def kennzeichen_ab(wort, i):
    """Gibt alle Kennzeichen zurück, die im Wort ab Stelle i passen, als Liste von (ende, kuerzel, mitte)."""
    ergebnis = []
    for laenge in range(1, laengste + 1):
        k = wort[i:i + laenge]
        if len(k) == laenge and k in kuerzel:             # ein Kürzel beginnt hier
            for mlaenge in (1, 2):
                m = wort[i + laenge:i + laenge + mlaenge]
                if len(m) == mlaenge and ist_mitte(m):    # Mittelteil passt noch ins Wort
                    ergebnis.append((i + laenge + mlaenge, k, m))
    return ergebnis


print(kennzeichen_ab('BWINF', 0))
print(kennzeichen_ab('BWINF', 1))
print(kennzeichen_ab('BWINF', 2))
print(kennzeichen_ab('TIMO', 0))
print(kennzeichen_ab('ABCÖ', 0))

[(2, 'B', 'W'), (3, 'B', 'WI')]
[(3, 'W', 'I'), (4, 'W', 'IN'), (4, 'WI', 'N'), (5, 'WI', 'NF')]
[(5, 'IN', 'F')]
[]
[(2, 'A', 'B'), (3, 'A', 'BC'), (3, 'AB', 'C')]


Bei `BWINF` gibt es ab Stelle 0 zwei Kennzeichen: `B-W` (reicht bis Stelle 2) und `B-WI` (bis Stelle 3). Ab Stelle 1 sind es vier (`W` und `WI` sind Kürzel), ab Stelle 2 eines (`IN-F`, das Wort endet danach). Bei `TIMO` beginnt kein Kürzel am Anfang, also gibt es kein Kennzeichen. Bei `ABCÖ` darf der Mittelteil kein `Ö` enthalten: `A-B`, `A-BC` und `AB-C` sind erlaubt, aber `AB-CÖ` und `A-BCÖ` nicht.

## Schritt 3: Die Tabelle füllen

Jetzt die Tabelle aus der Idee. Wir merken uns in `tabelle[e]` die Kennzeichen als Texte `KÜRZEL-MITTE`.

In [ ]:
def baue_tabelle(wort):
    """Gibt die Tabelle zurück: tabelle[i] ist eine Liste von Kennzeichen für die ersten i Buchstaben, sonst None."""
    n = len(wort)
    tabelle = [None] * (n + 1)
    tabelle[0] = []                                       # null Buchstaben: null Kennzeichen
    for i in range(n):
        if tabelle[i] is None:                            # bis hier kommt man nicht
            continue
        for ende, k, m in kennzeichen_ab(wort, i):
            if tabelle[ende] is None:                     # erste gefundene Möglichkeit genügt
                tabelle[ende] = tabelle[i] + [k + '-' + m]
    return tabelle


def zerlege(wort):
    """Gibt eine Liste von Kennzeichen zurück, die zusammen das Wort ergeben, oder None."""
    return baue_tabelle(wort)[len(wort)]


for i, eintrag in enumerate(baue_tabelle('BWINF')):
    print(i, eintrag)
print(zerlege('BWINF'))

0 []
1 None
2 ['B-W']
3 ['B-WI']
4 None
5 ['B-W', 'IN-F']
['B-W', 'IN-F']


Das ist die Tabelle wie in den Lösungshinweisen (dort mit einer etwas anderen Aufteilung): Stelle 0 ist leer, bei Stelle 2 und 3 stehen `B-W` und `B-WI`, Stelle 1 und 4 sind unmöglich (kein Kennzeichen endet dort), und bei Stelle 5 ist das ganze Wort `BWINF` mit `B-W` und `IN-F` geschrieben.

## Schritt 4: Alle Beispielwörter (Teilaufgaben 3 und 4)

Das Programm gibt zu jedem Wort die Aufteilung aus oder, dass keine möglich ist. Zur Kontrolle prüft eine zweite Funktion die Aufteilung: Setzt man Kürzel und Mittelteile aneinander, muss wieder das Wort herauskommen, jedes Kürzel muss in der Liste stehen und jeder Mittelteil aus ein bis zwei Buchstaben ohne Umlaute bestehen.

In [ ]:
def pruefe_aufteilung(wort, kennzeichen):
    """True, wenn die Liste von Kennzeichen 'KÜRZEL-MITTE' zusammen genau das Wort ergibt und alle erlaubt sind."""
    text = ''
    for kz in kennzeichen:
        k, m = kz.split('-')
        if k not in kuerzel or not (1 <= len(m) <= 2) or not ist_mitte(m):
            return False
        text = text + k + m
    return text == wort


for wort in woerter + ['HEIKE', 'TIMO']:
    liste = zerlege(wort)
    if liste is None:
        print(wort, '->', 'nicht möglich')
    else:
        assert pruefe_aufteilung(wort, liste)
        print(wort, '->', ' '.join(liste))

BIBER -> B-I B-ER
BUNDESWETTBEWERB -> B-U ND-ES WE-TT B-E WE-RB
CLINTON -> C-LI NT-ON
DONAUDAMPFSCHIFFFAHRTSKAPITÄNSMÜTZE -> nicht möglich
ETHERNET -> E-T H-E RN-ET
INFORMATIK -> IN-FO R-M AT-IK
LLANFAIRPWLLGWYNGYLLGOGERYCHWYRNDROBWLLLLANTYSILIOGOGOGOCH -> L-LA NF-AI RP-WL LG-WY N-GY LL-GO GER-YC H-WY R-ND R-OB WL-LL LAN-TY S-I LI-OG OG-OG OC-H
RINDFLEISCHETIKETTIERUNGSÜBERWACHUNGSAUFGABENÜBERTRAGUNGSGESETZ -> nicht möglich
SOFTWARE -> SO-FT WA-RE
TRUMP -> TR-U M-P
TSCHÜSS -> nicht möglich
VERKEHRSWEGEPLANUNGSBESCHLEUNIGUNGSGESETZ -> VER-KE HR-SW EG-EP LAN-UN GS-BE SC-HL EU-NI G-UN GS-GE SE-TZ
HEIKE -> HEI-KE
TIMO -> nicht möglich


Die Ergebnisse sind andere Aufteilungen als in den Lösungshinweisen (dort z. B. `B-U ND-ES WE-TT B-E WER-B`, hier steht am Ende `WE-RB`), aber gleichwertig: Ob ein Wort geht, hängt nicht davon ab. Bei den Wörtern **mit Umlauten** gibt es keine Aufteilung für `DONAUDAMPFSCHIFFFAHRTSKAPITÄNSMÜTZE`, `RINDFLEISCHETIKETTIERUNGSÜBERWACHUNGSAUFGABENÜBERTRAGUNGSGESETZ` und `TSCHÜSS`, denn im Mittelteil sind Umlaute verboten, und ein Kürzel mit `Ä` gibt es nicht (Kürzel mit `Ö` oder `Ü` gibt es dagegen, z. B. `GÖ` und `TÜ`; die Liste enthält sie, unser Programm findet sie wie alle anderen). Schreibt man die Umlaute als `UE`, `AE`, `OE` aus, geht es dagegen oft:

In [ ]:
for wort in ['TSCHUESS', 'DONAUDAMPFSCHIFFFAHRTSKAPITAENSMUETZE', 'RINDFLEISCHETIKETTIERUNGSUEBERWACHUNGSAUFGABENUEBERTRAGUNGSGESETZ']:
    liste = zerlege(wort)
    print(wort if len(wort) < 40 else wort[:30] + '...', '->', 'nicht möglich' if liste is None else ' '.join(liste))

TSCHUESS -> TS-CH UE-SS
DONAUDAMPFSCHIFFFAHRTSKAPITAENSMUETZE -> D-O NAU-D AM-PF SC-HI F-FF A-HR TS-KA PI-TA EN-SM UE-T Z-E
RINDFLEISCHETIKETTIERUNGSUEBER... -> nicht möglich


Die Umschreibung hilft bei `TSCHUESS` und `DONAUDAMPF…MUETZE`, aber nicht beim langen `RINDFLEISCH…`: Dort scheitert es an anderer Stelle. Warum, sehen wir uns gleich an.

## Schritt 5: Warum scheitert RINDFLEISCH…?

Wo bricht die Tabelle ab? Wir suchen die größte Stelle `i`, die noch erreichbar ist.

In [ ]:
lang = 'RINDFLEISCHETIKETTIERUNGSUEBERWACHUNGSAUFGABENUEBERTRAGUNGSGESETZ'
tab = baue_tabelle(lang)
letzte = max(i for i in range(len(lang) + 1) if tab[i] is not None)
print('Erreichbar bis Stelle', letzte, 'von', len(lang))
print('Das Wort bis dahin:', lang[:letzte], '|', lang[letzte:])
print('Aufteilung bis dahin:', ' '.join(tab[letzte]))

Erreichbar bis Stelle 18 von 65
Das Wort bis dahin: RINDFLEISCHETIKETT | IERUNGSUEBERWACHUNGSAUFGABENUEBERTRAGUNGSGESETZ
Aufteilung bis dahin: R-I ND-FL EI-SC HE-TI KE-TT


Ab der Stelle, an der die Tabelle endet, kann kein Kennzeichen mehr anfangen. Wir prüfen die Anfänge des Rests auf Kürzel:

In [ ]:
rest = lang[letzte:]
for laenge in range(1, 4):
    print(repr(rest[:laenge]), 'ist Kürzel:', rest[:laenge] in kuerzel)

'I' ist Kürzel: False
'IE' ist Kürzel: False
'IER' ist Kürzel: False


## Teilaufgaben 1 und 2: Welche Wörter gehen nicht?

**TIMO:** Das erste Kennzeichen beginnt immer am Wortanfang. Dort muss also ein Kürzel stehen: `T`, `TI` oder `TIM` (Kürzel haben höchstens drei Buchstaben). Wir fragen die Liste:

In [ ]:
for laenge in range(1, 4):
    print('TIMO'[:laenge], 'ist Kürzel:', 'TIMO'[:laenge] in kuerzel)

T ist Kürzel: False
TI ist Kürzel: False
TIM ist Kürzel: False


Keines davon ist ein Kürzel, also kann kein Kennzeichen mit `TIMO` beginnen. **Heike hat Recht, Timo nicht.**

Mit demselben Argument sind alle Wörter unmöglich, deren Anfang `X`, `XY` und `XYZ` kein Kürzel ist. Welche **einzelnen Buchstaben** sind kein Kürzel?

In [ ]:
kein_einzelkuerzel = [b for b in BUCHSTABEN if b not in kuerzel]
print(kein_einzelkuerzel)

['I', 'O', 'Q', 'T', 'U', 'X', 'Y']


Beispiele aus der Aufgabe und den Lösungshinweisen: `TIER` (kein `TI`, kein `TIE`), `TAT` (weder `T` noch `TA`), `SÜD` (kein `SÜ`) und die Zweibuchstabenwörter `IN` (kein `I`) und `UM` (kein `U`). Wir prüfen sie mit unserem Programm und zählen dann **alle** unmöglichen Wörter mit zwei und drei Buchstaben. Mit `itertools.product(BUCHSTABEN, repeat=2)` bekommt man alle Buchstabenpaare (eine Schleife statt zwei geschachtelter).

In [ ]:
import itertools

for wort in ['TIMO', 'TIER', 'TAT', 'IN', 'UM', 'SÜD']:
    print(wort, zerlege(wort))

for laenge in (2, 3):
    unmoeglich = []
    for tupel in itertools.product(BUCHSTABEN, repeat=laenge):
        wort = ''.join(tupel)
        if zerlege(wort) is None:
            unmoeglich.append(wort)
    print(laenge, 'Buchstaben:', len(unmoeglich), 'von', 26 ** laenge, 'Wörtern unmöglich, z. B.', unmoeglich[:6], '...', unmoeglich[-3:])

TIMO None
TIER None
TAT None
IN None
UM None
SÜD None
2 Buchstaben: 182 von 676 Wörtern unmöglich, z. B. ['IA', 'IB', 'IC', 'ID', 'IE', 'IF'] ... ['YX', 'YY', 'YZ']
3 Buchstaben: 3978 von 17576 Wörtern unmöglich, z. B. ['IAA', 'IAB', 'IAC', 'IAD', 'IAE', 'IAF'] ... ['YZX', 'YZY', 'YZZ']


Bei zwei Buchstaben sind es genau `7 · 26 = 182`: Ein Zweibuchstabenwort besteht aus einem Kennzeichen mit einem einbuchstabigen Kürzel und einem Buchstaben als Mittelteil, also scheitert es genau dann, wenn der erste Buchstabe (`I`, `O`, `Q`, `T`, `U`, `X`, `Y`) kein Kürzel ist. Auch mit **drei** Buchstaben gibt es unmögliche Wörter, z. B. `TAT`. Wörter mit **einem** Buchstaben gehen nie: Ein Kennzeichen hat mindestens zwei Buchstaben.

Ein Vierbuchstabenwort muss nicht gleich am Anfang scheitern. Wir suchen unter den Wörtern, die mit `B` beginnen, einige, die trotz des Kürzels `B` nicht gehen:

In [ ]:
beispiele = []
for tupel in itertools.product(BUCHSTABEN, repeat=3):
    wort = 'B' + ''.join(tupel)
    if zerlege(wort) is None:
        beispiele.append(wort)
print(len(beispiele), 'von', 26 ** 3, 'Wörtern BXXX unmöglich, z. B.', beispiele[:8])

2002 von 17576 Wörtern BXXX unmöglich, z. B. ['BDIA', 'BDIB', 'BDIC', 'BDID', 'BDIE', 'BDIF', 'BDIG', 'BDIH']


Nehmen wir `BDIA`: Mit `B-D` bliebe `IA`, aber `I` ist kein Kürzel; mit `B-DI` bliebe nur `A` übrig, und ein einzelner Buchstabe ist kein Kennzeichen. Wörter sind also nicht nur wegen ihres Anfangs unmöglich, sondern auch, weil am Ende ein Rest bleibt oder in der Mitte kein Kürzel passt.

## Für Neugierige: wenigste Kennzeichen und Anzahl der Aufteilungen

Mit derselben Tabellenidee lassen sich zwei weitere Fragen beantworten, wenn man statt der Liste eine Zahl in die Tabelle schreibt.

- **Wie wenige Kennzeichen braucht man mindestens?** `wenig[e]` ist die kleinste Anzahl Kennzeichen für die ersten `e` Buchstaben. Von jeder erreichbaren Stelle `i` aus gilt: `wenig[e] = min(wenig[e], wenig[i] + 1)`. Die offiziellen Hinweise nennen das „höchstens k Kennzeichen“.
- **Auf wie viele Arten geht es?** `arten[e]` ist die Anzahl der Aufteilungen der ersten `e` Buchstaben: `arten[e] = arten[e] + arten[i]`. Dabei zählen `AB-C` und `A-BC` als verschieden, weil es verschiedene Kennzeichen sind.

In [ ]:
def wenigste_und_anzahl(wort):
    """Gibt (kleinste Zahl Kennzeichen, Anzahl der Aufteilungen) zurück, oder (None, 0), wenn das Wort nicht geht."""
    n = len(wort)
    wenig = [None] * (n + 1)
    arten = [0] * (n + 1)
    wenig[0] = 0
    arten[0] = 1
    for i in range(n):
        if wenig[i] is None:
            continue
        for ende, k, m in kennzeichen_ab(wort, i):
            if wenig[ende] is None or wenig[i] + 1 < wenig[ende]:
                wenig[ende] = wenig[i] + 1
            arten[ende] = arten[ende] + arten[i]
    return wenig[n], arten[n]


print('Wort                     Buchst.  wenigste  Aufteilungen')
for wort in woerter:
    wenigste, anzahl = wenigste_und_anzahl(wort)
    print('%-24s %7d %9s %13d' % (wort[:24], len(wort), wenigste if wenigste is not None else '-', anzahl))

Wort                     Buchst.  wenigste  Aufteilungen
BIBER                          5         2             4
BUNDESWETTBEWERB              16         5            80
CLINTON                        7         2             1
DONAUDAMPFSCHIFFFAHRTSKA      35         -             0
ETHERNET                       8         3            12
INFORMATIK                    10         3             1
LLANFAIRPWLLGWYNGYLLGOGE      58        16      28871904
RINDFLEISCHETIKETTIERUNG      63         -             0
SOFTWARE                       8         2             7
TRUMP                          5         2             1
TSCHÜSS                        7         -             0
VERKEHRSWEGEPLANUNGSBESC      41        10       4875639


Je länger das Wort, desto mehr Aufteilungen gibt es, bei den längsten Wörtern Millionen. Eine Suche, die **alle** Aufteilungen ausprobiert, würde dadurch sehr langsam, die Tabelle zählt sie dagegen mit wenigen Rechenschritten. (Wörter ohne Aufteilung haben `0` Aufteilungen.) Die Beobachtung aus den Hinweisen: Für die Frage „geht es überhaupt?“ genügt *eine* Aufteilung pro Stelle.

## Zum Schluss: Dasselbe mit MiniZinc

Ist die Aufgabe mit MiniZinc lösbar? Ja, als **Erfüllbarkeitsproblem** („finde irgendeine Aufteilung“). Wir haben es bei **allen zwölf Beispielwörtern** ausprobiert: Mit dem Solver **Chuffed** dauert jedes unter einer halben Sekunde, auch der Beweis, dass es bei `RINDFLEISCH…` und den anderen unmöglichen Wörtern keine Aufteilung gibt. Der Standardsolver **Gecode** braucht bei dem langen walisischen Ortsnamen dagegen etwa 25 Sekunden und gibt bei `RINDFLEISCH…` nach 30 Sekunden auf. Das ist ein schönes Beispiel dafür, dass die Solverwahl wichtig ist.

**Das Modell:** Ein Wort mit `n` Buchstaben hat höchstens `n div 2` Kennzeichen, weil jedes mindestens zwei Buchstaben braucht. Für jedes mögliche Kennzeichen `p` gibt es Variablen:

- `benutzt[p]`: wird dieses Kennzeichen gebraucht? (Es werden immer die ersten gebraucht, die Bedingung `benutzt[p+1] -> benutzt[p]` verhindert Lücken.)
- `wahl[p]`: welches Kürzel aus der Tabelle (Zeilennummer),
- `mitte[p]`: Länge des Mittelteils, 1 oder 2,
- `start[p]`: Stelle im Wort, an der das Kennzeichen beginnt. `start[p+1]` ist `start[p]` plus die Länge des Kennzeichens (Kürzel plus Mittelteil), wenn es benutzt wird, sonst gleich. Am Ende muss `start` genau `n + 1` sein, damit das **ganze** Wort abgedeckt ist.

Die Bedingungen für ein benutztes Kennzeichen sagen: Jeder Buchstabe des Kürzels (`kz[wahl[p], q]`) steht im Wort an der richtigen Stelle (**Variable als Index**, wie bei den Arbeitsblättern), und die ein oder zwei Buchstaben des Mittelteils sind keine Umlaute (Code höchstens 26).

Die Kürzel stehen als 2D-Array `kz` (je Zeile ein Kürzel, mit `0` aufgefüllt) und als Längenliste `kl`. Buchstaben werden als Zahlen codiert: `A` = 1, …, `Z` = 26, `Ä` = 27, `Ö` = 28, `Ü` = 29 (einige Kürzel enthalten Umlaute, z. B. `GÖ` und `TÜ`). Das Modell (`autoscrabble.mzn`):

```minizinc
int: n;                                    % Länge des Wortes
int: anz;                                  % Anzahl der Kürzel
array[1..n] of int: w;                     % Buchstaben des Wortes als Zahlen (A = 1, ..., Z = 26, Ä = 27, Ö = 28, Ü = 29)
array[1..anz, 1..3] of int: kz;            % Kürzel als Zahlen, mit 0 aufgefüllt
array[1..anz] of int: kl;                  % Länge jedes Kürzels

int: maxk = n div 2;                       % so viele Kennzeichen höchstens
array[1..maxk] of var bool: benutzt;       % die ersten Kennzeichen werden benutzt
array[1..maxk] of var 1..anz: wahl;        % welches Kürzel
array[1..maxk] of var 1..2: mitte;         % Länge des Mittelteils
array[1..maxk+1] of var 1..n+1: start;     % Stelle, an der das Kennzeichen beginnt

constraint start[1] = 1;
constraint forall(p in 1..maxk)(
  start[p+1] = start[p] + (if benutzt[p] then kl[wahl[p]] + mitte[p] else 0 endif));
constraint start[maxk+1] = n + 1;          % das ganze Wort ist abgedeckt
constraint forall(p in 1..maxk-1)(benutzt[p+1] -> benutzt[p]);
constraint forall(p in 1..maxk)(benutzt[p] -> (
  forall(q in 1..3)(q <= kl[wahl[p]] -> w[start[p] + q - 1] = kz[wahl[p], q]) /\
  forall(m in 1..2)(m <= mitte[p] -> w[start[p] + kl[wahl[p]] + m - 1] <= 26)));
solve satisfy;

array[1..26] of string: buchstabe = ["A","B","C","D","E","F","G","H","I","J","K","L","M",
                                     "N","O","P","Q","R","S","T","U","V","W","X","Y","Z"];
output [ if fix(benutzt[p]) then
           concat([buchstabe[fix(kz[fix(wahl[p]), q])] | q in 1..fix(kl[fix(wahl[p])])]) ++ "-" ++
           concat([buchstabe[fix(w[fix(start[p]) + fix(kl[fix(wahl[p])]) + m - 1])] | m in 1..fix(mitte[p])]) ++ " "
         else "" endif | p in 1..maxk ] ++ ["\n"];
```

Die Daten (`.dzn`) erzeugt eine Python-Zelle aus den Beispieldaten. Die Kürzel-Tabelle ist für alle Wörter gleich, nur `n` und `w` ändern sich. Ausgabe für `TRUMP` (die Kürzel-Tabelle kürzen wir):

In [ ]:
def code(buchstabe):
    """A bis Z als 1 bis 26, Ä, Ö, Ü als 27, 28, 29 (auch einige Kürzel enthalten Umlaute)."""
    return (BUCHSTABEN + 'ÄÖÜ').index(buchstabe) + 1


def kuerzel_dzn():
    """Die Kürzel-Tabelle als MiniZinc-Daten."""
    liste = sorted(kuerzel)
    zahlen = []
    for k in liste:
        for q in range(3):
            zahlen.append(code(k[q]) if q < len(k) else 0)
    return ('anz = ' + str(len(liste)) + ';\nkz = array2d(1..' + str(len(liste)) + ', 1..3, ' + str(zahlen) + ');\n'
            + 'kl = ' + str([len(k) for k in liste]) + ';')


def wort_dzn(wort):
    """Das Wort als MiniZinc-Daten."""
    return 'n = ' + str(len(wort)) + ';\nw = ' + str([code(b) for b in wort]) + ';'


print(wort_dzn('TRUMP'))
print(kuerzel_dzn()[:150], '...')

n = 5;
w = [20, 18, 21, 13, 16];
anz = 684;
kz = array2d(1..684, 1..3, [1, 0, 0, 1, 1, 0, 1, 2, 0, 1, 2, 7, 1, 2, 9, 1, 3, 0, 1, 5, 0, 1, 8, 0, 1, 9, 2, 1, 9, 3, 1, 11, 0, 1, 12, 6, 1 ...


Ergebnisse (Solver Chuffed, jedes Wort in weniger als einer Sekunde; lange Wörter hinten gekürzt):

```
BIBER -> BI-B E-R
BUNDESWETTBEWERB -> B-UN DE-SW E-TT B-E WER-B
CLINTON -> C-LI NT-ON
DONAUDAMPFSCHIFFFAHR... -> nicht möglich
ETHERNET -> E-T H-E R-N E-T
INFORMATIK -> IN-FO R-M AT-IK
LLANFAIRPWLLGWYNGYLL... -> LL-A NF-AI RP-WL LG-WY N-GY LL-GO GER-YC H-WY RN-DR OB-WL LL-LA N-TY SI-LI OG-OG OG-O C-H
RINDFLEISCHETIKETTIE... -> nicht möglich
SOFTWARE -> SO-FT WA-RE
TRUMP -> TR-U M-P
TSCHÜSS -> nicht möglich
VERKEHRSWEGEPLANUNGS... -> VER-KE HR-SW EG-EP LAN-UN GS-BE SC-HL EU-NI G-UN GS-GE SE-TZ
TIMO -> nicht möglich
HEIKE -> HEI-KE
```

Die Aufteilungen sind gültig (kontrolliert mit unserer Funktion `pruefe_aufteilung`), aber nicht dieselben wie bei unserem Python-Programm: Das Modell sucht irgendeine Lösung. Die unmöglichen Wörter erkennt MiniZinc als `UNSATISFIABLE`, das Modell liefert hier genau dieselbe Antwort wie die Tabelle.

**Vergleich:** Das Modell ist kurz und beschreibt nur, *wie ein Kennzeichen aussieht* und dass sie nahtlos das Wort überdecken. Die Tabelle aus Python ist dagegen für lange Wörter immer schnell, braucht keinen Solver, und sie liefert die Fragen „wie wenige?“ und „wie viele Aufteilungen?“ gleich mit.